# SEASTRONG coastal toolkit — Colab runner

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alerovere/seastrong-field-tools/blob/main/SDB_toolkit/notebooks/seastrong_toolkit_colab.ipynb)

Documentation: [seastrong-field-tools.readthedocs.io → SDB toolkit](https://seastrong-field-tools.readthedocs.io/en/latest/sdb_toolkit.html)

The processing code lives in the Python package **`seastrong_toolkit`** (GitHub); this notebook only installs
it, loads your settings file and runs the steps. Each step can be re-run on its own:

| Step | What it does |
|---|---|
| `waves` | full ERA5 (waves, wind, rain) and Copernicus Marine wave records at a point |
| `prescreen` | suggests the Sentinel-2 composite months from waves, wind, rain, clouds and turbidity |
| `s2` | Sentinel-2 composite for Bathymetrix-AI (date screening, tile matching) |
| `dem`, `habitats`, `transport` | elevation, habitat layers, water-motion tracers |
| `icesat2` | ICESat-2 ATL24 bathymetry QC, locked validation tracks, SDB depth limit, ATL08 mangroves |

**Settings** are in one YAML file in your Drive (`CONFIG` below). Every step has its own settings hash and Drive
folder: change a value and re-run the step — only that step (and the steps that use its outputs) is
recomputed; unchanged steps re-use their finished exports.

**Credentials (Colab Secrets, key icon on the left):** `EE_PROJECT`, optional `EE_SERVICE_ACCOUNT_JSON`,
`CDS_KEY` (optional `CDS_URL`), `CMEMS_USERNAME`, `CMEMS_PASSWORD`. Nothing secret is stored in the notebook,
the settings file or the outputs.

In [ ]:
# ---- where the toolkit and your settings are (edit once)
REPO = 'https://github.com/Alerovere/seastrong-field-tools.git'   # public repository
SUBDIR = 'SDB_toolkit'                                           # folder of the package inside the repository
BRANCH = 'main'
CONFIG = '/content/drive/MyDrive/SEASTRONG/config_belize_south.yaml'   # your settings (created from the example if missing)
DEV_CLONE = None   # e.g. '/content/drive/MyDrive/SEASTRONG/seastrong-field-tools': clone to Drive, install editable, edit the .py files

## Install
Colab ships its own `google-auth`, `google-colab`, `earthengine-api` and related packages, and its sign-in, Drive
and Secrets depend on those exact versions. The cell below writes them into a **constraints file**, so pip
installs only what is missing and never replaces them (this removes the `google-auth` conflict). The versions
actually used are recorded in every step's `manifest.json`.

In [ ]:
import subprocess, sys, re, os
# 1. protect Colab's own packages (google-auth, google-colab, earthengine-api, ...) with a constraints file
frozen = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout.splitlines()
protect = re.compile(r'^(google-|googleapis-|grpcio|protobuf|proto-plus|earthengine-api|httplib2|ipykernel|ipython|'
                     r'jupyter|notebook|tornado|requests|urllib3)', re.I)
with open('/content/colab_constraints.txt', 'w') as f:
    f.write('\n'.join(l for l in frozen if '==' in l and protect.match(l)))
# 2. get the repository (a clone works for any folder name, also with spaces)
src = DEV_CLONE or '/content/seastrong-field-tools'
if not os.path.exists(src):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, src], check=True)
elif not DEV_CLONE:
    subprocess.run(['git', '-C', src, 'pull', '-q'], check=True)          # latest version of the toolkit
pkg = os.path.join(src, SUBDIR)
if not os.path.exists(os.path.join(pkg, 'pyproject.toml')):
    raise FileNotFoundError(f'No pyproject.toml in {pkg}: check SUBDIR / BRANCH')
# 3. install (editable when developing from a Drive clone)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *(['-e'] if DEV_CLONE else []), pkg,
                    '-c', '/content/colab_constraints.txt'], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError('Installation failed (see above). Do not remove the constraints file: it protects Colab sign-in.')

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Automatic re-loading of edited .py files. Colab's IPython autoreload still imports the 'imp' module, which
# Python 3.12 removed ("No module named 'imp'"): if it fails we skip it; use st.reload() after editing instead.
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception as e:
    print(f'autoreload not available ({e}); after editing the package run: st.reload(); ctx = st.setup(CONFIG, login=False, mount_drive=False)')

import seastrong_toolkit as st
print('seastrong_toolkit', st.__version__)
if not os.path.exists(CONFIG):
    st.write_example_config(CONFIG)
    raise SystemExit(f'Edit the example settings in {CONFIG} (AOI, PREFIX, wave POINT, ...), then run this cell again.')
ctx = st.setup(CONFIG, mount_drive=False)   # Earth Engine log-in (Colab Secrets), AOI, grid; prints each step's folder
ctx

**After editing the settings file**, reload it without logging in again:
`ctx = st.setup(CONFIG, login=False, mount_drive=False)`.

**After editing the package code** (with `DEV_CLONE`): `st.reload()`, then the same `st.setup(...)` line.

## 0. Waves, wind and rain (ERA5 + Copernicus Marine)

In [ ]:
st.waves.run(ctx)

## 0b. Pre-screening: which months for the Sentinel-2 composite?
Look at the two figures and the ranking, choose the months, write them into `s2: MONTHS:` in your settings
file, then reload the settings (cell above) before running step 1.

In [ ]:
st.prescreen.run(ctx)
st.prescreen.WINDOWS[st.prescreen.WINDOWS.rank_in_length <= 3][['length', 'label', 'usable_mean', 'usable_std', 'usable_min', 'usable_max', 'usable_per_month', 'p_good_day']]

## 1. Sentinel-2 composite (Bathymetrix-AI input)

In [ ]:
st.s2.run(ctx)

## 2–4. Elevation, habitats, transport overview

In [ ]:
st.dem.run(ctx)
st.habitats.run(ctx)

In [ ]:
st.transport.run(ctx)    # long exports (hours)

## 5. ICESat-2

In [ ]:
st.icesat2.run(ctx)

## Status of everything run in this session

In [ ]:
st.summary(ctx)